# Starter Notebook: Qwen 2B LoRA for Text-to-SVG (Kaggle)

This starter is built from the resources in `contest_docs`:
- Data resources: `contest_docs/03_Data_Design.md`
- Baseline and starter guidance: `contest_docs/05_Baselines_and_Starter_Notebooks.md`
- Kaggle implementation notes: `contest_docs/06_Kaggle_Implementation_Guide.md`

Goal: provide a practical scaffold for Qwen-2B-class fine-tuning + submission generation.

## Referenced Data and Docs

### Dataset resources from `contest_docs/03_Data_Design.md`
- `OmniSVG/MMSVG-Icon`
- `xingxm/SVGX-Core-250k`
- `xingxm/SVGX-SFT-1M` (recommended subset: `SVGX_SFT_GEN_51k.json`)
- `nyuuzyou/svgfind`
- `starvector/svg-icons`
- `thesantatitan/deepseek-svg-dataset`
- `InternSVG/SArena` (evaluation benchmark)

### Qwen 2B fine-tuning references from `contest_docs/05` and `contest_docs/06`
- Unsloth Qwen fine-tune docs: https://unsloth.ai/docs/models/qwen3.5/fine-tune
- Qwen3.5-2B Vision notebook: https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(2B)_Vision.ipynb

Note: this notebook is written as a reusable starter. You may need to adjust exact model IDs and column names to match the latest upstream datasets.

In [ ]:
# Uncomment in a fresh Kaggle notebook environment.
%pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas lxml huggingface_hub
from huggingface_hub import notebook_login
import unsloth
from unsloth import unsloth_train
notebook_login()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch

from datasets import concatenate_datasets, load_dataset, Dataset
import datasets

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Core training config.
# Keep runtime targets in line with contest_docs guidance (roughly <= 6-8 hours training).
CONFIG = {
    "model_name": "unsloth/Qwen3.5-2B",  # Verify exact ID from the linked Unsloth notebook.
    "max_seq_length": 8192,
    "lora_r": 64,
    "lora_alpha": 64,
    "learning_rate": 2e-4,
    "num_train_epochs": 1,
    "per_device_train_batch_size": 8,
    "gradient_accumulation_steps": 2,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "logging_steps": 10,
    "eval_steps": 100,
    "save_steps": 200,
    "max_train_samples_per_source": 50000,
    "eval_size": 0.02,
    "output_dir": "/content/drive/MyDrive/qwen2b_64x64_clean_drop_LR2_8192_weight_decay_0.01_last",
}

CONFIG

In [ ]:
SYSTEM_PROMPT = (
    "You generate compact, valid SVG markup from user requests. "
    "Return only SVG code with a single root <svg> element."
)


def format_sft_text(example):
    text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['prompt']}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{example['svg']}<|im_end|>"
    )
    return {"text": text}




In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=False,
)


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0.0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

## Kaggle Submission Inference Setup

**Instructions for Kaggle:**
1. Download the `qwen2b_svg_lora` folder generated by this training notebook.
2. Go to Kaggle, click **Create -> New Dataset**, and upload that folder. Name it something like `my-qwen2b-svg-lora`.
3. Create a **New Notebook** on Kaggle for your submission.
4. Click **Add Input** on the right sidebar and attach your new dataset.
5. Copy and paste the code below into your Kaggle submission notebook. Make sure to update `LORA_WEIGHTS_PATH` to match your dataset's path!

#If you wish to run inference on an existing model state, plug in model folder to MODEL_PATH variable

In [ ]:
# SETUP
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import time
import re
import xml.etree.ElementTree as ET
from tqdm.auto import tqdm

import torch
from unsloth import FastLanguageModel
from peft import PeftModel

MODEL_PATH = "/content/drive/MyDrive/qwen2b_64x64_clean_LR2_8192_weight_decay_0.01_last/checkpoint-3063"
TEST_PROMPTS_PATH = "/content/drive/MyDrive/test.csv"
SUBMISSION_PATH = "/content/drive/MyDrive/submission2.csv"

print("Saved files:", os.listdir(MODEL_PATH))

for name, module in model.named_modules():
  if "lora" in name.lower():
    print(name)
    break

In [ ]:

model, tokenizer = FastLanguageModel.from_pretrained(
  model_name=CONFIG["model_name"],
  max_seq_length=3072,
  dtype=None,
  load_in_4bit=False,
)

model = PeftModel.from_pretrained(model, MODEL_PATH)
FastLanguageModel.for_inference(model)

print("Model type:", type(model))
print("First parameter device:", next(model.parameters()).device)
print("Model + LoRA loaded")

FastLanguageModel.for_inference(model)

SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", flags=re.IGNORECASE)


def extract_svg(text):
    m = SVG_REGEX.search(text)
    return m.group(0).strip() if m else ""


def is_valid_svg(svg_text):
    if not svg_text:
        return False
    try:
        root = ET.fromstring(svg_text)
        return root.tag.endswith("svg")
    except ET.ParseError:
        return False


def fallback_svg(_prompt):
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="black"/>'
        '</svg>'
    )


def generate_svg(prompt, max_new_tokens=1024):
    chat_text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{prompt}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )
    inputs = tokenizer(text=[chat_text], return_tensors="pt").to(model.device)
    # Measure how many tokens were in your prompt
    input_length = inputs.input_ids.shape[1]
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.1,
            top_p=0.9,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    decoded_new = tokenizer.decode(output_ids[0][input_length:], skip_special_tokens=True)
    # decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    svg = extract_svg(decoded_new)

    # if not is_valid_svg(svg):
    #     print("failed")
    #     svg = fallback_svg(prompt)

    return svg


#test_prompt = "a simple blue bird icon"
#test_prompt = "The image features two orange squares with a microphone icon and an arrow connecting them, set against a white background."
# test_prompt = "Give me a red square"
# pred_svg = generate_svg(test_prompt)
# print(pred_svg)
# print("Valid SVG:", is_valid_svg(pred_svg))



test_df = pd.read_csv(TEST_PROMPTS_PATH)


rows = []
invalid_count = 0
t0 = time.time()

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Generating SVGs"):
    svg = generate_svg(row["prompt"])
    print(svg)
    if not is_valid_svg(svg):
        invalid_count += 1
        print(f"current count: {invalid_count}. failed svg: {svg}")
        svg = fallback_svg(row["prompt"])

    rows.append({"id": row["id"], "svg": svg})


sub_df = pd.DataFrame(rows)
sub_df.to_csv(SUBMISSION_PATH, index=False)

elapsed_min = (time.time() - t0) / 60

print("\nDONE")
print(f"Saved: {SUBMISSION_PATH}")
print(f"Rows: {len(sub_df)}")
print(f"Invalid/fallback count: {invalid_count}")
print(f"Runtime (minutes): {elapsed_min:.2f}")

## Notes

- Keep a fixed seed, runtime logs, and invalid-generation counts (required by `contest_docs/05`).
- If you use Kaggle-packaged datasets (`svg-train-public`, `svg-test-public-prompts`, `svg-utils`), swap paths into the loading cells.
- For stricter alignment with Unsloth templates, copy the latest prompt-formatting snippets from the official Qwen3.5-2B notebook linked above.